# 03 — Separar y filtrar autores UNAM

Se conserva el flujo de la versión anterior: separación por estructura de la fuente,
comparación de la misma publicación y consulta contextual de los archivos anteriores.

Cambios de esta versión:

- revisión por tamaño solo con **más de 20 autores O más de 20 afiliaciones**;
- no sumar autores y afiliaciones ni usar el número total de referencias como umbral;
- intentar resolver los casos pequeños antes de enviarlos a revisión;
- estar en una revisión antigua no obliga a seguir en revisión si ahora se resuelve;
- conservar únicamente afiliaciones UNAM del autor, una en cada columna (máximo dos);
- la normalización de nombres y denominaciones institucionales pertenece al **04**;
- los archivos anteriores son evidencia contextual, no listas globales de personas UNAM;
- no completar ni cambiar los otros doce campos, no deduplicar y no renumerar.

El notebook escribe solamente estos tres archivos en `04_Limpieza/01_Internos_unam`:

1. `autores_unam_separados_automatico.csv` — exactamente las 15 columnas originales.
2. `clasificacion_afiliaciones_unam.csv` — las tres columnas históricas.
3. `casos_revision_manual.csv` — registro original, motivo, propuestas y decisión.

Las comprobaciones y la trazabilidad detallada quedan en memoria y en las salidas
visibles del notebook, sin informes ni carpetas adicionales. Un caso ambiguo no se
convierte en externo por falta de datos. Los casos grandes siguen siendo manuales,
aunque exista una propuesta o solución histórica.

## Importaciones

In [1]:
import os
import re
import html
import csv
import io
import json
import hashlib
import unicodedata
from collections import defaultdict, Counter
from functools import lru_cache

import pandas as pd

## 1. RUTAS Y COLUMNAS

In [2]:
archivo = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"

archivo_tutora_xlsx = "../00_control/UNAM_Completo_Corregido.xlsx"
archivo_tutora_csv = "../00_control/UNAM_Completo_Corregido.csv"

carpeta_salida = "../04_Limpieza/01_Internos_unam"
archivo_clasificacion = f"{carpeta_salida}/clasificacion_afiliaciones_unam.csv"
revision_manual = f"{carpeta_salida}/casos_revision_manual.csv"
salida_automatica = f"{carpeta_salida}/autores_unam_separados_automatico.csv"

# Evidencia opcional ya existente. Se lee, pero NO se sobrescribe ni se concatena.
archivo_resueltos = f"{carpeta_salida}/casos_revision_resueltos.csv"
archivo_separados_anterior = f"{carpeta_salida}/autores_unam_separados.csv"

# Funciona desde notebooks/ o desde la raíz del repositorio en VS Code.
if not os.path.exists(archivo) and os.path.isdir("02_modelo_canonico"):
    archivo = archivo[3:]
    archivo_tutora_xlsx = archivo_tutora_xlsx[3:]
    archivo_tutora_csv = archivo_tutora_csv[3:]
    carpeta_salida = carpeta_salida[3:]
    archivo_clasificacion = archivo_clasificacion[3:]
    revision_manual = revision_manual[3:]
    salida_automatica = salida_automatica[3:]
    archivo_resueltos = archivo_resueltos[3:]
    archivo_separados_anterior = archivo_separados_anterior[3:]

LIMITE_ELEMENTOS = 20
usar_evidencia_historica = True
# Contrato del exportador IEEE empleado en la versión original: listas paralelas.
# Se comprueba la longitud, la estructura y las contradicciones con otra evidencia.
# Esta regla NO se aplica a las demás fuentes por simple igualdad de longitudes.
usar_listas_paralelas_ieee = True

# False: no reemplazar un CSV existente con contenido distinto.
# True: actualizar SOLO los tres CSV de salida. Guarda antes tu estado en GitHub Desktop.
actualizar_archivos = True

columnas = [
    "Fuente_origen", "indice", "Titulo", "Año", "Autor_norm",
    "Afiliacion1", "Afiliacion2", "ISBN", "ISSN", "Doi",
    "URL", "Area", "SubArea", "Keywords", "Abstract"
]
columnas_clasificacion = ["Afiliacion_original", "Estado_UNAM", "Evidencia"]
columnas_revision = columnas + [
    "ID_revision", "Motivo_revision", "Propuestas_no_aprobadas", "Decision",
    "Posicion_autor", "Autor_UNAM", "Afiliacion1_UNAM", "Afiliacion2_UNAM",
    "Revision_completa", "Evidencia", "URL_evidencia", "Comentario"
]
columnas_decision = [
    "Decision", "Posicion_autor", "Autor_UNAM", "Afiliacion1_UNAM",
    "Afiliacion2_UNAM", "Revision_completa", "Evidencia", "URL_evidencia", "Comentario"
]

## 2. FUNCIONES GENERALES

In [3]:
def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def decodificar_html(texto, max_iter=10):
    """Decodifica HTML antes de separar por ';'."""
    texto = "" if texto is None else str(texto)

    for _ in range(max_iter):
        nuevo = html.unescape(texto)
        if nuevo == texto:
            break
        texto = nuevo

    return unicodedata.normalize("NFC", texto).strip()


def separar_punto_coma(texto):
    texto = decodificar_html(texto)
    return [x.strip() for x in texto.split(";") if x.strip()]


def sin_acentos(texto):
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


@lru_cache(maxsize=50000)
def normalizar_texto(texto):
    """Solo para comparar; no reemplaza el valor guardado."""
    texto = sin_acentos(decodificar_html(texto)).lower()
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return " ".join(texto.split())


@lru_cache(maxsize=50000)
def normalizar_doi(doi):
    doi = decodificar_html(doi).lower().strip()
    doi = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", doi)
    doi = re.sub(r"^doi:\s*", "", doi)
    return re.sub(r"\s+", "", doi)


def normalizar_titulo(titulo):
    return normalizar_texto(titulo)


def unicos_no_vacios(valores):
    resultado = []
    vistos = set()

    for valor in valores:
        valor = decodificar_html(valor)
        clave = decodificar_html(valor).casefold()

        if valor and clave and clave not in vistos:
            resultado.append(valor)
            vistos.add(clave)

    return resultado


def leer_csv(ruta, esquema=None):
    with open(ruta, "rb") as f:
        if f.read(100).startswith(b"version https://git-lfs.github.com/spec/"):
            raise ValueError(f"{ruta}: es un puntero Git LFS, no el CSV de datos.")
    tabla = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    adicionales = [c for c in tabla if c.startswith("Unnamed:")]
    for c in adicionales:
        if tabla[c].str.strip().ne("").any():
            raise ValueError(f"{ruta}: {c} contiene información; no se elimina.")
    if adicionales:
        print(f"AVISO: {ruta}: {len(adicionales)} columnas vacías ignoradas en memoria.")
        tabla = tabla.drop(columns=adicionales)
    if esquema is not None and list(tabla.columns) != esquema:
        raise ValueError(f"{ruta}: esquema inesperado: {list(tabla.columns)}")
    return tabla


def preparar_csv(tabla):
    if not all(isinstance(x, str) for x in tabla.to_numpy().ravel()):
        raise ValueError("Todos los valores de los CSV deben ser texto.")
    texto = tabla.to_csv(index=False, lineterminator="\n", quoting=csv.QUOTE_ALL)
    relectura = pd.read_csv(io.StringIO(texto), dtype=str, keep_default_na=False)
    if not relectura.equals(tabla.reset_index(drop=True)):
        raise ValueError("La serialización CSV cambia los datos o su esquema.")
    return texto.encode("utf-8-sig")


def guardar_csv(tabla, ruta):
    contenido = preparar_csv(tabla)
    if os.path.exists(ruta):
        existente = pd.read_csv(
            ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig"
        )
        if existente.equals(tabla.reset_index(drop=True)):
            print("Sin cambios; esquema y datos verificados:", ruta)
            return True
        if not actualizar_archivos:
            print("NO se sobrescribió:", ruta)
            print("Resultado nuevo en memoria. Para guardarlo, activa actualizar_archivos.")
            return False
    # No se generan temporales, copias comprimidas ni otros archivos.
    modo = "wb" if os.path.exists(ruta) else "xb"
    with open(ruta, modo) as f:
        f.write(contenido)
    relectura = pd.read_csv(
        ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig"
    )
    if list(relectura.columns) != list(tabla.columns) or relectura.shape != tabla.shape:
        raise ValueError(f"{ruta}: la relectura cambió las filas, columnas o su orden.")
    if not relectura.equals(tabla.reset_index(drop=True)):
        raise ValueError(f"{ruta}: la relectura no coincide con los datos originales.")
    print("Guardado y releído:", ruta, tabla.shape)
    return True


def firma_registro(fila):
    datos = [fila[c] for c in columnas]
    return hashlib.sha256(json.dumps(datos, ensure_ascii=False).encode("utf-8")).hexdigest()


def ruta_csv_o_txt(ruta):
    """Los .txt adjuntos contienen CSV; preferir .csv cuando ambos existan."""
    if os.path.exists(ruta):
        return ruta
    alternativa = os.path.splitext(ruta)[0] + ".txt"
    return alternativa if os.path.exists(alternativa) else None


@lru_cache(maxsize=50000)
def normalizar_anio_comparacion(valor):
    # Solo llave temporal: NO cambia 2024.0, vacíos ni otros valores guardados.
    x = str(valor).strip()
    return x[:-2] if re.fullmatch(r"\d{4}\.0", x) else x


def misma_publicacion(a, b):
    da, db = normalizar_doi(a["Doi"]), normalizar_doi(b["Doi"])
    ta, tb = normalizar_titulo(a["Titulo"]), normalizar_titulo(b["Titulo"])
    if not ta or ta != tb or (da and db and da != db):
        return False
    if da and db:
        return True
    ya, yb = normalizar_anio_comparacion(a["Año"]), normalizar_anio_comparacion(b["Año"])
    return bool(ya and yb and ya == yb)


@lru_cache(maxsize=30000)
def texto_con_posiciones(texto):
    """Texto temporal de comparación y posiciones en la cadena original."""
    salida, posiciones = [], []
    for i, ch in enumerate(texto):
        for c in sin_acentos(ch).casefold():
            if c.isascii() and c.isalnum():
                salida.append(c); posiciones.append(i)
            elif salida and salida[-1] != " ":
                salida.append(" "); posiciones.append(i)
    if salida and salida[-1] == " ":
        salida.pop(); posiciones.pop()
    return "".join(salida), posiciones


def localizar_fragmentos(texto, catalogo):
    """Coincidencias institucionales; conserva la grafía de la cadena actual."""
    nt, pos = texto_con_posiciones(texto)
    encontrados = []
    for inst in catalogo:
        ni = normalizar_texto(inst)
        if not ni:
            continue
        inicio = 0
        while True:
            k = nt.find(ni, inicio)
            if k < 0:
                break
            fin = k + len(ni)
            if (k == 0 or nt[k-1] == " ") and (fin == len(nt) or nt[fin] == " "):
                encontrados.append((pos[k], pos[fin-1] + 1))
            inicio = k + 1
    elegidos = []
    for a, b in sorted(set(encontrados), key=lambda x: (-(x[1]-x[0]), x[0])):
        if not any(a < y and b > x for x, y in elegidos):
            elegidos.append((a, b))
    return sorted(elegidos)


def separar_afiliaciones(texto):
    """Separar listas sin contar como institución un CP o una continuación postal."""
    piezas = re.split(r"[;|]", decodificar_html(texto))
    resultado = []
    for p in piezas:
        p = p.strip()
        if not p:
            continue
        postal = bool(re.match(r"^(?:\d{3,}|[A-Z]{1,2}\d[A-Z\d ]*[, ]|[A-Z]{2},)", p))
        if resultado and postal:
            resultado[-1] += "; " + p
        else:
            resultado.append(p)
    return resultado

## 3. REFERENCIAS NUMÉRICAS Y COMPARACIÓN DE NOMBRES

In [4]:
# Scopus Author ID: (57191896522)
SCOPUS_ID_RE = re.compile(r"\s*\((\d{7,12})\)\s*$")

# Referencias institucionales: (1), (1,2), (1, 2, 3)
REF_AUTOR_RE = re.compile(r"\s*\((\d{1,3}(?:\s*,\s*\d{1,3})*)\)\s*$")
MARCADOR_AFILIACION_RE = re.compile(r"(?<!\w)\((\d{1,3})\)\s*")


def quitar_scopus_id(autor):
    return SCOPUS_ID_RE.sub("", decodificar_html(autor)).strip()


def referencias_autor(autor):
    m = REF_AUTOR_RE.search(decodificar_html(autor))
    return [x.strip() for x in m.group(1).split(",")] if m else []


def quitar_referencias_autor(autor):
    return REF_AUTOR_RE.sub("", decodificar_html(autor)).strip()


def nombre_separado(autor, fuente):
    # Retirar un ID largo solo cuando el registro es Scopus.
    autor = quitar_scopus_id(autor) if fuente == "Scopus" else decodificar_html(autor)
    return quitar_referencias_autor(autor)


def mapa_afiliaciones_numeradas(afiliacion1, afiliacion2=""):
    """Devuelve catálogo y problemas: nunca sobrescribe números contradictorios."""
    resultado, problemas = {}, []
    textos = [decodificar_html(afiliacion1), decodificar_html(afiliacion2)]
    hay_catalogo = any(MARCADOR_AFILIACION_RE.search(t) for t in textos)
    for texto in textos:
        if not texto:
            continue
        encontrados = list(MARCADOR_AFILIACION_RE.finditer(texto))
        if not encontrados:
            if hay_catalogo:
                problemas.append("columna sin referencias junto a catálogo numerado")
            continue
        if texto[:encontrados[0].start()].strip(" ;"):
            problemas.append("texto sin referencia antes del catálogo")
        for i, m in enumerate(encontrados):
            fin = encontrados[i + 1].start() if i + 1 < len(encontrados) else len(texto)
            afiliacion = texto[m.end():fin].strip(" ;")
            ref = m.group(1)
            if not afiliacion:
                problemas.append("afiliación numerada vacía")
            if ref in resultado and resultado[ref] != afiliacion:
                problemas.append("referencia institucional con valores distintos")
            else:
                resultado[ref] = afiliacion
    return resultado, problemas



def expandir_umlaut(texto):
    for a, b in {"ä":"ae", "ö":"oe", "ü":"ue", "Ä":"Ae", "Ö":"Oe", "Ü":"Ue", "ß":"ss"}.items():
        texto = texto.replace(a, b)
    return sin_acentos(texto)


@lru_cache(maxsize=50000)
def variantes_nombre(nombre):
    nombre = quitar_referencias_autor(quitar_scopus_id(nombre))
    variantes = []
    for texto in [sin_acentos(nombre), expandir_umlaut(nombre)]:
        if "," in texto:
            apellidos, nombres = texto.split(",", 1)
            versiones = [nombres + " " + apellidos, apellidos + " " + nombres]
        else:
            versiones = [texto]
            # Scopus etiqueta sus entradas como Apellidos I.I., sin coma.
            m = re.match(r"^(.*?)\s+((?:[A-Z]\.\s*)+)$", texto.strip())
            if m:
                versiones.append(m.group(2) + " " + m.group(1))
        for v in versiones:
            tokens = re.findall(r"[^\W_]+", v.casefold())
            if tokens not in variantes:
                variantes.append(tokens)
    return variantes


def tokens_nombre(nombre):
    return variantes_nombre(nombre)[0] if variantes_nombre(nombre) else []


def token_compatible(a, b):
    return a == b or (len(a) == 1 and b.startswith(a)) or (len(b) == 1 and a.startswith(b))


def comparar_tokens(a, b):
    if min(len(a), len(b)) < 2:
        return False
    if len(a) == len(b) and all(token_compatible(x, y) for x, y in zip(a, b)):
        return any(x == y and len(x) > 1 for x, y in zip(a, b))
    corto, largo = (a, b) if len(a) < len(b) else (b, a)
    if not (token_compatible(corto[0], largo[0]) and corto[-1] == largo[-1] and len(corto[-1]) > 1):
        return False
    j = 0
    for token in corto:
        while j < len(largo) and not token_compatible(token, largo[j]):
            j += 1
        if j == len(largo):
            return False
        j += 1
    return True


@lru_cache(maxsize=50000)
def nombres_compatibles(a, b):
    # Solo dentro de la MISMA publicación. La unicidad se verifica en ambos lados.
    # No cambia el nombre de salida ni utiliza diccionarios de normalización.
    return any(comparar_tokens(x, y) for x in variantes_nombre(a) for y in variantes_nombre(b))


def coincidencias_nombre(nombre, lista):
    # Una coincidencia textual exacta no prevalece si otra persona es compatible.
    return [i for i, candidato in enumerate(lista) if nombres_compatibles(nombre, candidato)]

## 4. SCOPUS, WOS Y UNAM

In [5]:
# Se reutilizan las clasificaciones históricas, corrigiendo colisiones y cadenas mixtas.
UNAM_COMPLETA = re.compile(
    r"\b(?:universidad nacional autonoma de mexico|national autonomous university of mexico|"
    r"universite nationale autonome du mexique|univ nacl autonoma mexico|univ nacional autonoma de mexico)\b"
)
COLISION_UNAM = re.compile(
    r"\b(?:bilkent|universidad nacional de misiones|universidad nacional de moquegua|university of namibia)\b|"
    r"\bunam\b.*\b(?:obera|argentina|moquegua|namibia|turkey|turkiye)\b"
)
EXTERNAS_EXPLICITAS = re.compile(
    r"\b(?:bilkent|cinvestav|cimat|inaoe|inmegen|inria|cnrs|inserm|conicet|northeastern|"
    r"universidad autonoma de la ciudad de mexico|universidad autonoma metropolitana|"
    r"instituto politecnico nacional|national polytechnic institute|tecnologico de monterrey|"
    r"tecnologico nacional de mexico|instituto tecnologico de morelia|"
    r"instituto nacional de medicina genomica|national institute of genomic medicine|"
    r"instituto nacional de enfermedades respiratorias|hospital general de mexico|"
    r"instituto nacional de pediatria|consejo nacional de humanidades|"
    r"centre for research in mathematics|centro de investigacion en matematicas)\b"
)
UNIVERSIDAD = re.compile(r"\b(?:universidad|university|universite|universitat|universita|universidade|universiti|universiteit|univ)\b")
DEPENDENCIAS_UNAM = re.compile(
    r"\b(?:iimas|dgtic|instituto de investigaciones en matematicas aplicadas y en sistemas|"
    r"institute for applied mathematics and systems research|"
    r"centro de estudios en computacion avanzada|"
    r"direccion general de computo y de tecnologias de informacion y comunicacion)\b"
)
SIGLAS_UNAM = re.compile(r"\bunam\b|\bu n a m\b")
GENERICA_DEPENDENCIA = re.compile(r"^(?:facultad de (?:ingenieria|ciencias|medicina)|instituto de (?:fisica|matematicas)|department of computer science|facultad de ciencias de la computacion)$")
clasificacion_por_texto = defaultdict(list)
catalogo_atomico = []


@lru_cache(maxsize=50000)
def clasificar_literal(afiliacion):
    texto = normalizar_texto(afiliacion)
    if not texto:
        return "AMBIGUA", "afiliación vacía"
    if len(separar_afiliaciones(afiliacion)) > 1:
        return "MIXTA", "varias afiliaciones delimitadas; separarlas antes de filtrar"
    completa = bool(UNAM_COMPLETA.search(texto))
    if COLISION_UNAM.search(texto):
        if completa:
            return "MIXTA", "UNAM México y otra institución homónima en la misma cadena"
        return "EXTERNO", "institución distinta de UNAM México; colisión de siglas"
    resto = UNAM_COMPLETA.sub("", texto)
    resto = re.sub(r"\b(?:avenida|av|ave|avenue) universidad\b|\buniversity city\b", "", resto)
    siglas = bool(SIGLAS_UNAM.search(texto))
    otra = bool(EXTERNAS_EXPLICITAS.search(resto) or UNIVERSIDAD.search(resto))
    doble_dependencia = bool(re.search(r"\bfacultad\b.+\b(?:and|y)\s+instituto\b", texto))
    if completa:
        if otra or len(UNAM_COMPLETA.findall(texto)) > 1 or doble_dependencia:
            return "MIXTA", "varias instituciones o dependencias en una cadena"
        return "UNAM", "denominación explícita de la Universidad Nacional Autónoma de México"
    if siglas:
        if otra:
            return "MIXTA", "siglas UNAM junto a otra institución; delimitar"
        if re.search(r"\bmexico\b|\bcdmx\b", texto) or DEPENDENCIAS_UNAM.search(texto):
            return "UNAM", "siglas con contexto mexicano o dependencia UNAM identificable"
        return "AMBIGUA", "siglas sin contexto institucional suficiente"
    if DEPENDENCIAS_UNAM.search(texto) and not otra:
        return "UNAM", "dependencia específica incluida en las reglas anteriores"
    if otra:
        return "EXTERNO", "denominación de otra institución, sin afiliación UNAM explícita"
    return "AMBIGUA", "denominación que requiere clasificación contextual"


@lru_cache(maxsize=50000)
def clasificar_afiliacion(afiliacion):
    estado, evidencia = clasificar_literal(afiliacion)
    decisiones = clasificacion_por_texto.get(normalizar_texto(afiliacion), [])
    estados = {r["Estado_UNAM"] for r in decisiones if r["Estado_UNAM"]}
    if estado in {"MIXTA", "EXTERNO"}:
        return estado, evidencia
    if estado == "UNAM":
        conflicto_previo = any(r["Estado_UNAM"] == "AMBIGUA" and "contradic" in normalizar_texto(r["Evidencia"]) for r in decisiones)
        if conflicto_previo:
            return "AMBIGUA", "contradicción institucional pendiente de revisión explícita"
        if "EXTERNO" in estados:
            return "AMBIGUA", "contradicción entre clasificación previa y denominación institucional"
        return estado, evidencia
    if len(estados) > 1:
        return "AMBIGUA", "clasificaciones anteriores contradictorias"
    if decisiones:
        d = decisiones[0]
        if d["Estado_UNAM"] == "EXTERNO":
            return "EXTERNO", d["Evidencia"] + " | Clasificación institucional histórica reutilizada"
        # No convertir una dependencia genérica en UNAM solo por el control.
        if d["Estado_UNAM"] == "UNAM" and not GENERICA_DEPENDENCIA.fullmatch(normalizar_texto(afiliacion)):
            prueba = d["Evidencia"]
            if "http" in prueba or prueba.startswith("VERIFICADO:"):
                return "UNAM", prueba
        if d["Estado_UNAM"] in {"AMBIGUA", "MIXTA"}:
            return d["Estado_UNAM"], d["Evidencia"]
    return estado, evidencia


def es_afiliacion_unam(afiliacion):
    return clasificar_afiliacion(afiliacion)[0] == "UNAM"


@lru_cache(maxsize=15000)
def partir_afiliacion(afiliacion):
    """No traduce ni reemplaza nombres institucionales; extrae segmentos originales."""
    texto = decodificar_html(afiliacion)
    piezas = separar_afiliaciones(texto)
    if len(piezas) > 1:
        return [q for p in piezas for q in partir_afiliacion(p)]
    if not texto:
        return []
    if clasificar_literal(texto)[0] != "MIXTA":
        return [texto]
    # Se aprovechan cadenas atómicas YA presentes en el catálogo histórico.
    spans = localizar_fragmentos(texto, catalogo_atomico)
    if len(spans) > 1:
        resto = list(texto)
        for a, b in spans:
            resto[a:b] = " " * (b-a)
        sobrante = re.sub(r"\b(?:and|y)\b", "", "".join(resto), flags=re.I)
        if not sobrante.strip(" ,;.\t\n|"):
            return unicos_no_vacios([texto[a:b] for a, b in spans])
    # Límite explícito: país separado por comas + comienzo de otra institución.
    patron = re.compile(
        r",\s*(?:M[eé]xico|Spain|France|Germany|Japan|United States|Brazil|Argentina|Chile|"
        r"Portugal|Canada|United Kingdom|Peru|Colombia|Finland|Ireland|Norway|Sweden)\s*[.]?,\s*"
        r"(?=(?:Department|Departamento|Instituto|Institute|Centro|Center|Research|Facultad|Faculty|"
        r"Universidad|University|National|Escuela|School|Laboratorio|Laboratory|Unidad|UNAM)\b)", re.I
    )
    cortes = [m.end() for m in patron.finditer(texto)]
    if cortes:
        partes = [texto[a:b].strip(" ,;") for a, b in zip([0]+cortes, cortes+[len(texto)])]
        if all(clasificar_literal(p)[0] in {"UNAM", "EXTERNO"} for p in partes):
            return unicos_no_vacios(partes)
    return [texto]


def afiliaciones_scopus(entrada_autor, catalogo):
    entrada = decodificar_html(entrada_autor)
    spans = localizar_fragmentos(entrada, catalogo)
    if not spans:
        return "", [], False
    etiqueta = entrada[:spans[0][0]].strip(" ,;")
    resto = list(entrada[spans[0][0]:])
    for a, b in spans:
        resto[a-spans[0][0]:b-spans[0][0]] = " " * (b-a)
    completo = not "".join(resto).strip(" ,;.\t\n")
    return etiqueta, unicos_no_vacios([entrada[a:b] for a, b in spans]), completo


def corresponding_wos(texto):
    resultado = []
    for segmento in separar_afiliaciones(texto):
        m = re.fullmatch(r"(.*?)\s*\(corresponding author\)\s*,\s*(.+)", segmento, re.I | re.S)
        if m:
            resultado.append((m.group(1).strip(), m.group(2).strip()))
    return resultado

## 5. CARGAR DATOS

In [6]:
if not os.path.exists(archivo):
    raise FileNotFoundError(f"Falta la entrada original: {archivo}. No se sustituye por una salida posterior.")
hash_antes = sha256(archivo)
df = leer_csv(archivo, columnas)
for campo in ["indice", "ISBN", "ISSN"]:
    malos = df[campo].str.contains(r"(?<![A-Za-z0-9])\d+(?:[.,]\d+)?[eE][+-]?\d+(?![A-Za-z0-9])", regex=True)
    if malos.any():
        raise ValueError(f"{campo}: notación científica en la ENTRADA, filas {(df.index[malos]+1).tolist()[:20]}. No se reconstruyen dígitos.")
df["_row_id"] = range(len(df))
firmas = df.apply(firma_registro, axis=1).astype(str) if len(df) else pd.Series(index=df.index, dtype=str)
ocurrencias = firmas.groupby(firmas, sort=False).cumcount() + 1
df["_id_revision"] = firmas + "_" + ocurrencias.astype(str)

ruta_clasificacion = ruta_csv_o_txt(archivo_clasificacion)
clasificacion = leer_csv(ruta_clasificacion, columnas_clasificacion) if ruta_clasificacion else pd.DataFrame(columns=columnas_clasificacion)
vacias = clasificacion.apply(lambda c: c.str.strip().eq("")).all(axis=1)
if vacias.any():
    print("Filas completamente vacías ignoradas en la clasificación:", int(vacias.sum()))
    clasificacion = clasificacion.loc[~vacias].reset_index(drop=True)
if clasificacion["Afiliacion_original"].duplicated().any():
    raise ValueError("La clasificación contiene claves exactas duplicadas.")
if not set(clasificacion["Estado_UNAM"]) <= {"UNAM", "EXTERNO", "AMBIGUA", "MIXTA"}:
    raise ValueError("Estado institucional no permitido.")
clasificacion_por_texto = defaultdict(list)
for r in clasificacion.to_dict("records"):
    clasificacion_por_texto[normalizar_texto(r["Afiliacion_original"])].append(r)

# Si una cadena histórica EXTERNO contiene varias instituciones separadas,
# sus componentes explícitos conservan esa clasificación. No se propaga UNAM.
componentes_externos = []
for r in clasificacion.to_dict("records"):
    if r["Estado_UNAM"] != "EXTERNO":
        continue
    for bloque in separar_afiliaciones(r["Afiliacion_original"]):
        # Un país al final de una institución delimita el comienzo de otra.
        patron_limite = re.compile(
            r",\s*(?:M[eé]xico|Spain|France|Germany|Japan|United States|Brazil|Argentina|Chile|"
            r"Portugal|Canada|United Kingdom|Peru|Colombia|Finland|Ireland|Norway|Sweden|Costa Rica)\s*[.]?,\s*"
            r"(?=(?:Department|Departamento|Instituto|Institute|Centro|Center|Research|Facultad|Faculty|"
            r"Universidad|University|National|Escuela|School|Laboratorio|Laboratory|Unidad|The|Advanced)\b)", re.I
        )
        cortes = [m.end() for m in patron_limite.finditer(bloque)]
        partes = [bloque[a:b].strip(" ,;") for a,b in zip([0]+cortes,cortes+[len(bloque)])]
        for parte in partes:
            clave = normalizar_texto(parte)
            if clave and clave not in clasificacion_por_texto and not SIGLAS_UNAM.search(clave) and not UNAM_COMPLETA.search(clave) and not DEPENDENCIAS_UNAM.search(clave):
                e = {"Afiliacion_original":parte, "Estado_UNAM":"EXTERNO",
                    "Evidencia":"Componente de cadena clasificada EXTERNO en el archivo anterior: " + r["Afiliacion_original"]}
                componentes_externos.append(e)
                clasificacion_por_texto[clave].append(e)
clasificacion = pd.concat([clasificacion,pd.DataFrame(componentes_externos,columns=columnas_clasificacion)],ignore_index=True)

catalogo_atomico = unicos_no_vacios([
    r["Afiliacion_original"] for r in clasificacion.to_dict("records")
    if len(normalizar_texto(r["Afiliacion_original"])) >= 10
    and clasificar_literal(r["Afiliacion_original"])[0] in {"UNAM", "EXTERNO"}
])
partir_afiliacion.cache_clear()
clasificar_afiliacion.cache_clear()

columnas_tutora = ["Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2", "Doi"]
tutora = pd.DataFrame(columns=columnas_tutora)
ruta_tutora = ruta_csv_o_txt(archivo_tutora_csv)
if ruta_tutora:
    tutora = leer_csv(ruta_tutora)
elif os.path.exists(archivo_tutora_xlsx):
    tutora = pd.read_excel(archivo_tutora_xlsx, dtype=str, keep_default_na=False)
if not set(columnas_tutora) <= set(tutora.columns):
    raise ValueError("El control no contiene las columnas necesarias.")

# Se cargan antes de escribir, únicamente como evidencia de autor y afiliación.
# No se copian sus ISBN/ISSN, no se asume que sean independientes entre sí,
# no se deduce que un autor omitido en estas tablas sea externo.
# La salida automática que se regenerará NO es un donante: evita ciclos entre ejecuciones.
historicos = []
if usar_evidencia_historica:
    for ruta, origen in [
        (archivo_resueltos, "REVISION_ANTERIOR"),
        (archivo_separados_anterior, "SEPARADOS_ANTERIOR")
    ]:
        ruta_real = ruta_csv_o_txt(ruta)
        if ruta_real:
            h = leer_csv(ruta_real, columnas)
            h = h[columnas_tutora + ["Fuente_origen", "indice"]].copy()
            h["_origen_evidencia"] = origen
            historicos.append(h)
evidencia_historica = pd.concat(historicos, ignore_index=True) if historicos else pd.DataFrame(columns=columnas_tutora + ["Fuente_origen", "indice", "_origen_evidencia"])

revision_anterior = pd.DataFrame(columns=columnas_revision)
ruta_revision = ruta_csv_o_txt(revision_manual)
if ruta_revision:
    anterior = leer_csv(ruta_revision)
    if list(anterior.columns) == columnas:
        # Una lista antigua de pendientes NO fuerza la nueva revisión.
        print("Casos históricos leídos como referencia, no como revisión obligatoria:", len(anterior))
    elif list(anterior.columns) == columnas_revision:
        seleccion = []
        for i, fila in anterior.iterrows():
            firma = firma_registro(fila)
            posibles = df.loc[firmas.eq(firma), "_id_revision"].tolist()
            tiene_decision = fila["Decision"] not in {"", "PENDIENTE"} or any(fila[c].strip() for c in columnas_decision if c != "Decision")
            if fila["ID_revision"] in posibles:
                seleccion.append(fila.to_dict())
            elif len(posibles) == 1 and not fila["ID_revision"]:
                d = fila.to_dict(); d["ID_revision"] = posibles[0]; seleccion.append(d)
            elif tiene_decision:
                raise ValueError(f"Decisión manual de otra versión, fila {i+1}; no se sobrescribe.")
        revision_anterior = pd.DataFrame(seleccion, columns=columnas_revision)
    else:
        raise ValueError("El archivo de casos manuales tiene un esquema no reconocido.")

print("Filas de entrada:", len(df))
print("Columnas canónicas:", len(columnas))
print("Filas del control:", len(tutora))
print("Relaciones históricas disponibles como evidencia:", len(evidencia_historica))
print("SHA-256 de la entrada:", hash_antes)

Filas de entrada: 2582
Columnas canónicas: 15
Filas del control: 5255
Relaciones históricas disponibles como evidencia: 0
SHA-256 de la entrada: 194ba501cd22a158dbcf227c9087a0dc1068c33d5ffeb322973ade6525da1512


## 6. CASOS GRANDES: >20 AUTORES O >20 AFILIACIONES

In [7]:
def contar_estructura(fila):
    autores = separar_punto_coma(fila["Autor_norm"])
    catalogo, _ = mapa_afiliaciones_numeradas(fila["Afiliacion1"], fila["Afiliacion2"])
    fuente = fila["Fuente_origen"].strip()
    if catalogo:
        # No contar ni el valor máximo de la referencia ni cuántos autores la usan.
        n_afiliaciones = sum(max(1, len(partir_afiliacion(a))) for a in catalogo.values())
    elif fuente in {"Scopus", "WoS"} and fila["Afiliacion1"].strip():
        # Afiliacion2 describe enlaces/correspondencia: no duplicar el catálogo.
        # Contar entradas del catálogo, no sus menciones en cada autor.
        n_afiliaciones = len([
            p for a in separar_afiliaciones(fila["Afiliacion1"]) for p in partir_afiliacion(a)
        ])
    else:
        n_afiliaciones = len([
            p for c in ["Afiliacion1", "Afiliacion2"]
            for a in separar_afiliaciones(fila[c]) for p in partir_afiliacion(a)
        ])
    return pd.Series({
        "_n_autores": len(autores), "_n_afiliaciones": n_afiliaciones,
        "_n_referencias": sum(len(referencias_autor(a)) for a in autores)
    })

conteos = df.apply(contar_estructura, axis=1) if len(df) else pd.DataFrame(columns=["_n_autores", "_n_afiliaciones", "_n_referencias"])
df = pd.concat([df, conteos], axis=1)
df["_caso_grande"] = df[["_n_autores", "_n_afiliaciones"]].gt(LIMITE_ELEMENTOS).any(axis=1)
print("Casos con más de", LIMITE_ELEMENTOS, "autores O afiliaciones:", int(df["_caso_grande"].sum()))
print("Las referencias repetidas no activan la revisión por tamaño.")

Casos con más de 20 autores O afiliaciones: 34
Las referencias repetidas no activan la revisión por tamaño.


## 7. PRIMERA PASADA: SEPARACIÓN AUTOR-AFILIACIÓN

In [8]:
problemas_por_fila = defaultdict(list)


def asignar_relacion(relacion, afiliaciones, evidencia, tipo):
    afiliaciones = unicos_no_vacios([p for a in afiliaciones for p in partir_afiliacion(a)])
    if afiliaciones:
        relacion["Afiliaciones"] = afiliaciones
        relacion["Estado_separacion"] = "COMPLETO"
        relacion["Evidencia"] = evidencia
        relacion["Tipo_vinculo"] = tipo


def separar_fila(fila):
    row_id = fila["_row_id"]
    fuente = fila["Fuente_origen"].strip()
    originales = separar_punto_coma(fila["Autor_norm"])
    autores = [nombre_separado(a, fuente) for a in originales]
    base = {"row_id": row_id, "Fuente_origen": fila["Fuente_origen"], "indice": fila["indice"],
        "Titulo": fila["Titulo"], "Año": fila["Año"], "Doi": fila["Doi"]}
    salida = [dict(base, Autor=a, Posicion_autor=str(i+1), Afiliaciones=[],
        Estado_separacion="PENDIENTE", Evidencia="sin relación explícita", Tipo_vinculo="")
        for i, a in enumerate(autores)]
    if not autores:
        problemas_por_fila[row_id].append("lista de autores vacía")
        return salida
    if fila["_caso_grande"]:
        return salida
    if len({normalizar_texto(a) for a in autores}) != len(autores):
        problemas_por_fila[row_id].append("nombres repetidos dentro del registro; desambiguar")
    if any(len(tokens_nombre(a)) < 2 or all(len(t) == 1 for t in tokens_nombre(a)) for a in autores):
        problemas_por_fila[row_id].append("nombre insuficiente para identificar a una persona")
    if any(not a or re.search(r"@|\bet\s+al\b|\b(?:collaboration|consortium)\b", a, re.I) for a in autores):
        problemas_por_fila[row_id].append("lista de autores incompleta o autor colectivo")

    catalogo, errores = mapa_afiliaciones_numeradas(fila["Afiliacion1"], fila["Afiliacion2"])
    if catalogo:
        # Solo los conflictos de numeración son definitivos. Una referencia ausente
        # puede aclararse en las pasadas siguientes con la MISMA publicación.
        if any("valores distintos" in e for e in errores):
            problemas_por_fila[row_id].append("referencias institucionales contradictorias")
        for original, relacion in zip(originales, salida):
            refs = referencias_autor(original)
            if not refs and len(autores) == len(catalogo) == 1 and not errores:
                refs = list(catalogo)
            if refs and not errores and all(x in catalogo and catalogo[x] for x in refs):
                asignar_relacion(relacion, [catalogo[x] for x in dict.fromkeys(refs)],
                    "referencias explícitas del autor: " + original, "NUMERADA")
        return salida
    if any(referencias_autor(a) for a in originales):
        return salida

    if fuente == "Scopus":
        catalogo = separar_afiliaciones(fila["Afiliacion1"])
        usadas = defaultdict(int)
        for entrada in separar_afiliaciones(fila["Afiliacion2"]):
            etiqueta, instituciones, completo = afiliaciones_scopus(entrada, catalogo)
            candidatos = coincidencias_nombre(etiqueta, autores)
            if completo and len(candidatos) == 1:
                i = candidatos[0]
                usadas[i] += 1
                if usadas[i] == 1:
                    asignar_relacion(salida[i], instituciones, "Scopus: etiqueta nominal única + catálogo; " + entrada, "SCOPUS_NOMINAL")
                else:
                    salida[i]["Estado_separacion"] = "PENDIENTE"
                    salida[i]["Afiliaciones"] = []
                    problemas_por_fila[row_id].append("Scopus: varias entradas corresponden a una misma aparición de autor")
        return salida

    if fuente == "WoS":
        for etiqueta, institucion in corresponding_wos(fila["Afiliacion2"]):
            candidatos = coincidencias_nombre(etiqueta, autores)
            if len(candidatos) == 1:
                r = salida[candidatos[0]]
                asignar_relacion(r, r["Afiliaciones"] + [institucion],
                    "WoS: corresponding author identificado nominalmente; " + etiqueta, "WOS_NOMINAL")
        return salida

    if fuente == "IEEE" and usar_listas_paralelas_ieee:
        afiliaciones = separar_afiliaciones(fila["Afiliacion1"])
        # Recupera el contrato de las exportaciones IEEE usado en el código original.
        # Afiliacion2 con información adicional invalida esta interpretación sencilla.
        if afiliaciones and len(autores) == len(afiliaciones) and not fila["Afiliacion2"].strip():
            for r, a in zip(salida, afiliaciones):
                asignar_relacion(r, [a], "IEEE: correspondencia posicional del formato original de exportación", "IEEE_PARALELA")
            return salida

    if len(autores) == 1:
        # Registro individual, no un catálogo multiautor. No se extiende a listas
        # institucionales grandes ni a fuentes con estructuras nominales propias.
        afiliaciones = unicos_no_vacios([p for c in ["Afiliacion1", "Afiliacion2"] for p in partir_afiliacion(fila[c])])
        if 1 <= len(afiliaciones) <= LIMITE_ELEMENTOS:
            asignar_relacion(salida[0], afiliaciones, "registro individual con afiliaciones explícitas", "INDIVIDUAL")
    return salida


registros = []
for _, fila in df.iterrows():
    registros.extend(separar_fila(fila))
columnas_relaciones = ["row_id", "Fuente_origen", "indice", "Titulo", "Año", "Doi", "Autor",
    "Posicion_autor", "Afiliaciones", "Estado_separacion", "Evidencia", "Tipo_vinculo"]
relaciones = pd.DataFrame(registros, columns=columnas_relaciones)
print("Apariciones de autor extraídas:", len(relaciones))
print("Primera pasada:")
print(relaciones["Estado_separacion"].value_counts())

Apariciones de autor extraídas: 14996
Primera pasada:
Estado_separacion
COMPLETO     9846
PENDIENTE    5150
Name: count, dtype: int64


## 8. SEGUNDA PASADA: MISMA PUBLICACIÓN EN OTRA FUENTE

In [9]:
# Se restaura la recuperación contextual del código anterior, SIN fuzzy global.
# Solo se usa evidencia ya resuelta; no hay cascadas de copias entre filas pendientes.
propuestas_por_fila = defaultdict(list)
conflictos_contextuales = defaultdict(list)
nombres_por_fila = {i: g["Autor"].tolist() for i, g in relaciones.groupby("row_id", sort=False)}


def clave_afiliacion_comparacion(a):
    texto = UNAM_COMPLETA.sub("unam", normalizar_texto(a))
    texto = re.sub(r"instituto de investigaciones en matematicas aplicadas y en sistemas|institute for applied mathematics and systems research", "iimas", texto)
    return " ".join(texto.split())


def instituciones_entrada(row_id):
    fila = df.loc[row_id]
    cat, _ = mapa_afiliaciones_numeradas(fila["Afiliacion1"], fila["Afiliacion2"])
    if cat:
        valores = cat.values()
    elif fila["Fuente_origen"].strip() in {"Scopus", "WoS"}:
        valores = separar_afiliaciones(fila["Afiliacion1"])
    else:
        valores = [fila["Afiliacion1"], fila["Afiliacion2"]]
    return unicos_no_vacios([p for a in valores for p in partir_afiliacion(a)])


catalogos_por_fila = {i: instituciones_entrada(i) for i in df.index}


def ajustar_al_catalogo(afiliaciones, catalogo, exigir=False):
    resultado = []
    for afiliacion in afiliaciones:
        a = clave_afiliacion_comparacion(afiliacion)
        candidatos = [c for c in catalogo if clave_afiliacion_comparacion(c) == a]
        if not candidatos:
            candidatos = [c for c in catalogo if clave_afiliacion_comparacion(c)
                and (a.startswith(clave_afiliacion_comparacion(c) + " ")
                     or clave_afiliacion_comparacion(c) in a)]
            candidatos = [c for c in candidatos if clasificar_afiliacion(c)[0] == clasificar_afiliacion(afiliacion)[0]]
        if len(candidatos) == 1:
            resultado.append(candidatos[0])
        elif exigir:
            return None
        else:
            resultado.append(afiliacion)
    return unicos_no_vacios(resultado)


def firma_unam(afiliaciones):
    estados = [clasificar_afiliacion(a)[0] for a in afiliaciones]
    if not estados or any(e not in {"UNAM", "EXTERNO"} for e in estados):
        return None
    unam = frozenset(clave_afiliacion_comparacion(a) for a, e in zip(afiliaciones, estados) if e == "UNAM")
    return ("UNAM", unam) if unam else ("EXTERNO", frozenset())


def indice_evidencias(evidencias):
    por_doi, por_titulo = defaultdict(list), defaultdict(list)
    for e in evidencias:
        d = normalizar_doi(e["Doi"])
        t = normalizar_titulo(e["Titulo"])
        if d:
            por_doi[d].append(e)
        if t:
            por_titulo[t].append(e)
    return por_doi, por_titulo


def candidatos_contextuales(r, indices):
    por_doi, por_titulo = indices
    doi, titulo = normalizar_doi(r["Doi"]), normalizar_titulo(r["Titulo"])
    candidatos = por_doi.get(doi, []) if doi else por_titulo.get(titulo, [])
    # Si el DOI no tiene donantes, título+año puede enlazar un registro sin DOI.
    if not candidatos:
        candidatos = por_titulo.get(titulo, [])
    resultado = []
    for e in candidatos:
        if e.get("row_id") == r["row_id"] or not misma_publicacion(r, e):
            continue
        if not nombres_compatibles(r["Autor"], e["Autor"]):
            continue
        if len(coincidencias_nombre(e["Autor"], nombres_por_fila[r["row_id"]])) != 1:
            continue
        if e.get("row_id") is not None and len(coincidencias_nombre(r["Autor"], nombres_por_fila[e["row_id"]])) != 1:
            continue
        resultado.append(e)
    return resultado


def resolver_con_evidencias(i, candidatos, exigir_catalogo=False):
    r = relaciones.loc[i]
    if df.loc[r["row_id"], "_caso_grande"] or not candidatos:
        return False
    grupos = defaultdict(list)
    for e in candidatos:
        afiliaciones = ajustar_al_catalogo(e["Afiliaciones"], catalogos_por_fila[r["row_id"]], exigir=exigir_catalogo)
        if afiliaciones is None:
            continue
        firma = firma_unam(afiliaciones)
        if firma is not None:
            grupos[firma].append((e, afiliaciones))
    if len(grupos) > 1:
        conflictos_contextuales[r["row_id"]].append("evidencia contextual contradictoria para " + r["Autor"])
        return False
    if not grupos:
        return False
    e, afiliaciones = next(iter(grupos.values()))[0]
    relaciones.at[i, "Afiliaciones"] = afiliaciones
    relaciones.at[i, "Estado_separacion"] = "COMPLETO"
    relaciones.at[i, "Tipo_vinculo"] = e.get("Tipo_vinculo", "EVIDENCIA_CONTEXTUAL")
    relaciones.at[i, "Evidencia"] = "Misma publicación y autor inequívoco | " + e["Evidencia"]
    return True


# Copia fija: las recuperaciones no se vuelven donantes en esta pasada.
anclas = [r for r in relaciones.to_dict("records") if r["Estado_separacion"] == "COMPLETO"
    and not problemas_por_fila.get(r["row_id"]) and firma_unam(r["Afiliaciones"]) is not None]
indices_anclas = indice_evidencias(anclas)
recuperadas_otras = 0
for i, r in relaciones.iterrows():
    if df.loc[r["row_id"], "_caso_grande"]:
        continue
    candidatos = candidatos_contextuales(r, indices_anclas)
    if r["Estado_separacion"] != "COMPLETO":
        recuperadas_otras += int(resolver_con_evidencias(i, candidatos))
    elif r["Tipo_vinculo"] in {"IEEE_PARALELA", "INDIVIDUAL"}:
        # El orden no gana frente a un enlace nominal/numérico contradictorio.
        for e in candidatos:
            if e["Tipo_vinculo"] in {"NUMERADA", "SCOPUS_NOMINAL", "WOS_NOMINAL"}:
                otra = ajustar_al_catalogo(e["Afiliaciones"], catalogos_por_fila[r["row_id"]])
                if firma_unam(otra) is not None and firma_unam(r["Afiliaciones"]) != firma_unam(otra):
                    conflictos_contextuales[r["row_id"]].append("la relación posicional/individual contradice otro enlace explícito: " + r["Autor"])
print("Relaciones recuperadas desde otra representación explícita:", recuperadas_otras)

Relaciones recuperadas desde otra representación explícita: 618


## 9. TERCERA PASADA: ARCHIVO DE LA TUTORA Y EVIDENCIA ANTERIOR

In [10]:
# Se reutiliza lo ya trabajado; no se decide por presencia del nombre en una lista.
# La tutora y las salidas automáticas antiguas necesitan además una afiliación
# compatible con el catálogo ORIGINAL del registro actual. Una revisión resuelta
# aporta una relación revisada, pero no acredita a los autores que no incluyó.
evidencias_previas = []
for _, e in evidencia_historica.iterrows():
    if ";" in e["Autor_norm"] or len(tokens_nombre(e["Autor_norm"])) < 2:
        continue
    afiliaciones = unicos_no_vacios([p for c in ["Afiliacion1", "Afiliacion2"] for p in partir_afiliacion(e[c])])
    if firma_unam(afiliaciones) is None:
        continue
    evidencias_previas.append({"Titulo":e["Titulo"], "Año":e["Año"], "Doi":e["Doi"],
        "Autor":e["Autor_norm"], "Afiliaciones":afiliaciones,
        "Tipo_vinculo":e["_origen_evidencia"],
        "Evidencia":e["_origen_evidencia"] + ": " + e["Fuente_origen"] + " / " + e["indice"]})
for _, e in tutora.iterrows():
    if ";" in e["Autor_norm"] or len(tokens_nombre(e["Autor_norm"])) < 2:
        continue
    afiliaciones = unicos_no_vacios([p for c in ["Afiliacion1", "Afiliacion2"] for p in partir_afiliacion(e[c])])
    if firma_unam(afiliaciones) is not None:
        evidencias_previas.append({"Titulo":e["Titulo"], "Año":e["Año"], "Doi":e["Doi"],
            "Autor":e["Autor_norm"], "Afiliaciones":afiliaciones,
            "Tipo_vinculo":"CONTROL_TUTORA", "Evidencia":"control tutora, misma publicación + autor + catálogo actual"})
indices_previos = indice_evidencias(evidencias_previas)
recuperadas_previas = 0
for i, r in relaciones.iterrows():
    candidatos = candidatos_contextuales(r, indices_previos)
    for e in candidatos:
        propuestas_por_fila[r["row_id"]].append(e["Evidencia"] + " | " + e["Autor"] + " | " + "; ".join(e["Afiliaciones"]))
    if df.loc[r["row_id"], "_caso_grande"] or r["Estado_separacion"] == "COMPLETO":
        continue
    revisados = [e for e in candidatos if e["Tipo_vinculo"] == "REVISION_ANTERIOR"]
    # No homologar dos personas distintas con una etiqueta abreviada común.
    nombres_candidatos = {tuple(tokens_nombre(e["Autor"])) for e in revisados}
    incompatibles = any(not nombres_compatibles(a["Autor"], b["Autor"]) for a in revisados for b in revisados)
    if incompatibles:
        conflictos_contextuales[r["row_id"]].append("identidades distintas compatibles con un nombre abreviado: " + r["Autor"])
        continue
    if revisados:
        # Si el registro dispone de instituciones, la revisión anterior no puede
        # introducir una UNAM ausente de ese catálogo ni cambiar de dependencia.
        exigir = bool(catalogos_por_fila[r["row_id"]])
        if resolver_con_evidencias(i, revisados, exigir_catalogo=exigir):
            recuperadas_previas += 1
            continue
    otros = [e for e in candidatos if e["Tipo_vinculo"] != "REVISION_ANTERIOR"]
    if otros and catalogos_por_fila[r["row_id"]]:
        recuperadas_previas += int(resolver_con_evidencias(i, otros, exigir_catalogo=True))

# Las alertas conocidas NO bloquean el artículo completo por su título.
# Solo un autor afectado y una asignación UNAM dudosa provocan revisión.
alertas_autor_publicacion = [
    ("Finding the Set of Nearly Optimal Solutions of a Multiobjective Optimization Problem", "C. Segura"),
    ("Screening and Structural Characterization of Heat Shock Response Elements (HSEs) in Entamoeba histolytica Promoters", "Elisa Azuara-Liceaga")
]
for _, r in relaciones.iterrows():
    for titulo, autor in alertas_autor_publicacion:
        if normalizar_titulo(r["Titulo"]) == normalizar_titulo(titulo) and nombres_compatibles(r["Autor"], autor):
            firma = firma_unam(r["Afiliaciones"])
            if firma is None or firma[0] != "EXTERNO":
                conflictos_contextuales[r["row_id"]].append("verificar asignación histórica UNAM del autor concreto: " + r["Autor"])
print("Relaciones recuperadas mediante evidencia histórica contextual:", recuperadas_previas)
print("No se cambiaron nombres ni metadatos bibliográficos.")

Relaciones recuperadas mediante evidencia histórica contextual: 0
No se cambiaron nombres ni metadatos bibliográficos.


## 10. FILAS QUE VAN A REVISIÓN MANUAL

In [11]:
motivos_revision = defaultdict(list)
for row_id, problemas in problemas_por_fila.items():
    motivos_revision[row_id].extend(problemas)
for row_id, problemas in conflictos_contextuales.items():
    motivos_revision[row_id].extend(problemas)
for _, fila in df.iterrows():
    if fila["_caso_grande"]:
        motivos_revision[fila["_row_id"]].append(
            f">{LIMITE_ELEMENTOS} autores o afiliaciones (autores={fila['_n_autores']}; afiliaciones={fila['_n_afiliaciones']})")

# Solo preservar decisiones ACTIVAS del formulario actual. Una fila PENDIENTE
# de la revisión antigua no fuerza manual si ahora se ha separado correctamente.
for _, anterior in revision_anterior.iterrows():
    if anterior["Decision"] not in {"", "PENDIENTE"} or any(anterior[c].strip() for c in columnas_decision if c != "Decision"):
        ids = df.loc[df["_id_revision"].eq(anterior["ID_revision"]), "_row_id"].tolist()
        for row_id in ids:
            motivos_revision[row_id].append("decisión manual existente que debe conservarse")

filas_pendientes, filas_mas_dos = set(), set()
clasificaciones_relacion = []
for _, r in relaciones.iterrows():
    estados = [clasificar_afiliacion(a)[0] for a in r["Afiliaciones"]]
    clasificaciones_relacion.append(estados)
    if df.loc[r["row_id"], "_caso_grande"]:
        continue
    if r["Estado_separacion"] != "COMPLETO" or not estados:
        filas_pendientes.add(r["row_id"])
        motivos_revision[r["row_id"]].append("separación autor-afiliación aún no resuelta después de las tres pasadas")
    if any(e in {"AMBIGUA", "MIXTA"} for e in estados):
        motivos_revision[r["row_id"]].append("institución o delimitación de afiliaciones ambigua")
    # Contar únicamente las afiliaciones UNAM, después de separar y retirar externas.
    unam = unicos_no_vacios([a for a, e in zip(r["Afiliaciones"], estados) if e == "UNAM"])
    if len(unam) > 2:
        filas_mas_dos.add(r["row_id"])
        motivos_revision[r["row_id"]].append("autor con más de dos afiliaciones UNAM")
relaciones["Clasificaciones"] = clasificaciones_relacion
motivos_revision = {i: sorted(set(v)) for i, v in motivos_revision.items() if v}
filas_revision = set(motivos_revision)
diagnostico_revision = pd.DataFrame([
    {"_row_id": i, "Motivo_revision": " | ".join(motivos_revision[i])} for i in sorted(filas_revision)
], columns=["_row_id", "Motivo_revision"])
print("Registros originales para revisión:", len(filas_revision))
print("  Por tamaño:", int(df["_caso_grande"].sum()))
print("  De hasta 20 elementos con ambigüedad persistente:", sum(not df.loc[i, "_caso_grande"] for i in filas_revision))

Registros originales para revisión: 663
  Por tamaño: 34
  De hasta 20 elementos con ambigüedad persistente: 629


## 11. CLASIFICAR UNAM / EXTERNO

In [12]:
def clasificar_unam(fila):
    if fila["row_id"] in filas_revision:
        return "REVISION"
    if fila["Estado_separacion"] != "COMPLETO" or not fila["Clasificaciones"]:
        return "REVISION"
    if "UNAM" in fila["Clasificaciones"]:
        return "UNAM"
    if all(e == "EXTERNO" for e in fila["Clasificaciones"]):
        return "EXTERNO"
    return "REVISION"

relaciones["Estado_UNAM"] = relaciones.apply(clasificar_unam, axis=1) if len(relaciones) else pd.Series(dtype=str)
relaciones_automaticas = relaciones[~relaciones["row_id"].isin(filas_revision)].copy()
# Auditoría disponible en memoria, sin un CSV adicional.
auditoria_relaciones = relaciones.copy(deep=True)
print("Clasificación publicación + autor:")
print(relaciones["Estado_UNAM"].value_counts())

Clasificación publicación + autor:
Estado_UNAM
REVISION    6554
EXTERNO     4593
UNAM        3849
Name: count, dtype: int64


## 12. CONSTRUIR SALIDA AUTOMÁTICA

In [13]:
relaciones_unam = relaciones_automaticas[relaciones_automaticas["Estado_UNAM"].eq("UNAM")].copy()
salida_trabajo = []
for _, r in relaciones_unam.iterrows():
    original = df.loc[r["row_id"], columnas].to_dict()
    afiliaciones = unicos_no_vacios([
        a for a, e in zip(r["Afiliaciones"], r["Clasificaciones"]) if e == "UNAM"
    ])
    if not 1 <= len(afiliaciones) <= 2:
        raise ValueError("Se intentó conservar un autor sin una o dos afiliaciones UNAM.")
    original["Autor_norm"] = r["Autor"]
    original["Afiliacion1"] = afiliaciones[0]
    original["Afiliacion2"] = afiliaciones[1] if len(afiliaciones) == 2 else ""
    original["_row_id"] = r["row_id"]
    salida_trabajo.append(original)

autores_unam_automatico = pd.DataFrame(salida_trabajo, columns=columnas + ["_row_id"])
# _row_id solo se usa en memoria para validar la procedencia de cada relación.
# La selección explícita excluye cualquier columna auxiliar del archivo automático.
salida_automatica_df = autores_unam_automatico.loc[:, columnas].copy().reset_index(drop=True)
print("Filas UNAM automáticas:", len(salida_automatica_df))

Filas UNAM automáticas: 3849


## 13. ÚNICO ARCHIVO DE REVISIÓN MANUAL

In [14]:
# Los 15 campos originales nunca se editan para resolver un caso.
# Decision: PENDIENTE / CONSERVAR / EXCLUIR.
# CONSERVAR: una copia de la fila por autor, Posicion_autor (1, 2, ...),
# Autor_UNAM, una o dos afiliaciones UNAM, Evidencia y URL_evidencia.
# Revision_completa=SI certifica que se revisó el registro entero y que
# los autores no incluidos se descartan de forma documentada (explicar en Comentario).
# EXCLUIR: una sola fila con motivo y evidencia; no clasifica a toda la persona como externa.
# PENDIENTE: no produce filas finales y no se interpreta como una exclusión aprobada.

casos = []
for i in sorted(filas_revision):
    fila = df.loc[i]
    anteriores = revision_anterior[revision_anterior["ID_revision"].eq(fila["_id_revision"])]
    if anteriores.empty:
        anteriores = pd.DataFrame([{c: "" for c in columnas_revision}])
        anteriores["Decision"] = "PENDIENTE"
    for _, anterior in anteriores.iterrows():
        registro = {c: fila[c] for c in columnas}
        registro["ID_revision"] = fila["_id_revision"]
        registro["Motivo_revision"] = " | ".join(motivos_revision[i])
        propuestas = list(dict.fromkeys(propuestas_por_fila.get(i, [])))
        registro["Propuestas_no_aprobadas"] = " || ".join(propuestas)
        for c in columnas_decision:
            registro[c] = anterior[c]
        casos.append(registro)
casos_revision = pd.DataFrame(casos, columns=columnas_revision).fillna("").astype(str)


# Conservar el catálogo anterior y agregar lo observado; no generar otro diccionario.
filas_clasificacion = []
for r in clasificacion.to_dict("records"):
    estado, evidencia = clasificar_afiliacion(r["Afiliacion_original"])
    if estado == r["Estado_UNAM"]:
        evidencia = r["Evidencia"]
    else:
        evidencia = "Corrección de " + r["Estado_UNAM"] + ": " + evidencia + " | Antecedente: " + r["Evidencia"]
    filas_clasificacion.append({"Afiliacion_original":r["Afiliacion_original"], "Estado_UNAM":estado, "Evidencia":evidencia})
conocidas = {r["Afiliacion_original"] for r in filas_clasificacion}
a_observar = [a for lista in relaciones["Afiliaciones"] for a in lista]
a_observar += [a for lista in catalogos_por_fila.values() for a in lista]
for a in unicos_no_vacios(a_observar):
    if a not in conocidas:
        estado, motivo = clasificar_afiliacion(a)
        filas_clasificacion.append({"Afiliacion_original":a, "Estado_UNAM":estado,
            "Evidencia":motivo + ". Clasificación institucional, no prueba del vínculo autor-publicación."})
        conocidas.add(a)
clasificacion_final = pd.DataFrame(filas_clasificacion, columns=columnas_clasificacion).fillna("").astype(str)
print("Registros originales distintos en revisión:", casos_revision["ID_revision"].nunique())

Registros originales distintos en revisión: 663


## 14. VALIDACIONES

In [15]:
columnas_inmutables = [c for c in columnas if c not in {"Autor_norm", "Afiliacion1", "Afiliacion2"}]


def validar_salida(tabla):
    if list(tabla.columns) != columnas + ["_row_id"]:
        raise ValueError("La tabla no conserva su esquema de trabajo.")
    for _, fila in tabla.iterrows():
        original = df.loc[fila["_row_id"]]
        if any(fila[c] != original[c] for c in columnas_inmutables):
            raise ValueError("Cambió un metadato protegido en la separación.")
        if not fila["Autor_norm"].strip() or ";" in fila["Autor_norm"]:
            raise ValueError("La salida debe tener exactamente un autor por fila.")
        if not fila["Afiliacion1"].strip():
            raise ValueError("Una fila conservada requiere al menos una afiliación UNAM.")
        if fila["Afiliacion2"] and decodificar_html(fila["Afiliacion1"]).casefold() == decodificar_html(fila["Afiliacion2"]).casefold():
            raise ValueError("Afiliacion1 y Afiliacion2 son idénticas.")


validar_salida(autores_unam_automatico)
for _, fila in autores_unam_automatico.iterrows():
    original = df.loc[fila["_row_id"]]
    nombres = [nombre_separado(a, original["Fuente_origen"].strip()) for a in separar_punto_coma(original["Autor_norm"])]
    if fila["Autor_norm"] not in nombres:
        raise ValueError("Se alteró un nombre; la normalización pertenece al código 04.")
    if original["_caso_grande"]:
        raise ValueError("Un caso de más de 20 elementos se incluyó como automático.")

if autores_unam_automatico["_row_id"].isin(filas_revision).any():
    raise ValueError("Un registro de revisión fue incluido también como automático.")
if len(columnas) != 15 or list(salida_automatica_df.columns) != columnas:
    raise ValueError("La salida automática debe tener exactamente las 15 columnas originales, en su orden.")
if any(c.startswith("Unnamed") or c.startswith("_") for c in salida_automatica_df.columns):
    raise ValueError("La salida automática contiene columnas auxiliares.")
if not salida_automatica_df.equals(autores_unam_automatico[columnas].reset_index(drop=True)):
    raise ValueError("La salida automática no coincide con las relaciones automáticas validadas.")
for campo in ["Afiliacion1", "Afiliacion2"]:
    for afiliacion in salida_automatica_df[campo]:
        if afiliacion and not es_afiliacion_unam(afiliacion):
            raise ValueError("La salida automática contiene una afiliación no confirmada como UNAM.")
if hash_antes != sha256(archivo):
    raise ValueError("Se modificó el archivo original durante el proceso.")
if list(clasificacion_final.columns) != columnas_clasificacion:
    raise ValueError("La clasificación tiene columnas adicionales.")
# Verifica en memoria la relectura exacta antes de cualquier escritura.
preparar_csv(clasificacion_final)
preparar_csv(casos_revision)
preparar_csv(salida_automatica_df)
print("Validaciones previas a escritura: OK")

Validaciones previas a escritura: OK


## 15. INCORPORACIÓN OPCIONAL DE LA REVISIÓN MANUAL

In [16]:
# Toda la revisión se resuelve dentro de casos_revision_manual.csv.
# No se requiere casos_revision_resueltos.csv ni otros formularios.
manual_aprobado, cierres_revision = [], []
ids_pendientes = set()
por_id = df.set_index("_id_revision", drop=False)
for identificador, grupo in casos_revision.groupby("ID_revision", sort=False):
    estados = set(grupo["Decision"])
    if not estados <= {"PENDIENTE", "CONSERVAR", "EXCLUIR"}:
        raise ValueError(f"{identificador}: Decision inválida.")
    if not set(grupo["Revision_completa"]) <= {"", "NO", "SI"}:
        raise ValueError(f"{identificador}: Revision_completa debe ser SI, NO o vacío.")
    if "PENDIENTE" in estados or not grupo["Revision_completa"].eq("SI").all():
        ids_pendientes.add(identificador)
        continue
    if len(estados) != 1:
        raise ValueError("Un registro cerrado no puede ser EXCLUIR y CONSERVAR a la vez.")
    original = por_id.loc[identificador]
    if any(grupo[c].ne(original[c]).any() for c in columnas):
        raise ValueError("Se alteraron los campos originales de la revisión.")
    for campo in ["Evidencia", "Comentario"]:
        if grupo[campo].str.strip().eq("").any():
            raise ValueError(f"Una revisión cerrada requiere {campo}.")
    if estados == {"EXCLUIR"}:
        if len(grupo) != 1 or any(grupo[c].str.strip().ne("").any() for c in ["Posicion_autor", "Autor_UNAM", "Afiliacion1_UNAM", "Afiliacion2_UNAM"]):
            raise ValueError("EXCLUIR se registra una vez, sin autores ni afiliaciones de salida.")
        cierres_revision.append({"ID_revision": identificador, "Decision": "EXCLUIR", "Filas": 0})
        continue
    nombres = [nombre_separado(a, original["Fuente_origen"].strip()) for a in separar_punto_coma(original["Autor_norm"])]
    usadas = set()
    for _, decision in grupo.iterrows():
        posicion = decision["Posicion_autor"].strip()
        if not re.fullmatch(r"[1-9]\d*", posicion) or not 1 <= int(posicion) <= len(nombres):
            raise ValueError("Posicion_autor debe identificar al autor en la lista original (desde 1).")
        if posicion in usadas:
            raise ValueError("Hay dos decisiones de conservación para la misma aparición de autor.")
        usadas.add(posicion)
        autor = decision["Autor_UNAM"]
        if autor != nombres[int(posicion)-1]:
            raise ValueError("Autor_UNAM debe ser el nombre separado original; normalizar nombres pertenece al 04.")
        if not re.fullmatch(r"https?://[^\s]+", decision["URL_evidencia"].strip()):
            raise ValueError("Una conservación manual exige URL de evidencia bibliográfica.")
        af1, af2 = decision["Afiliacion1_UNAM"], decision["Afiliacion2_UNAM"]
        if not af1.strip() or (af2 and decodificar_html(af1).casefold() == decodificar_html(af2).casefold()):
            raise ValueError("Indica una o dos afiliaciones UNAM distintas, una por columna.")
        for afiliacion in [af1, af2]:
            if not afiliacion:
                continue
            estado = clasificar_afiliacion(afiliacion)[0]
            if estado != "UNAM":
                raise ValueError("Afiliación manual no reconocida como UNAM: aporta denominación inequívoca o clasificación verificada.")
        fila = {c: original[c] for c in columnas}
        fila.update(Autor_norm=autor, Afiliacion1=af1, Afiliacion2=af2, _row_id=original["_row_id"])
        manual_aprobado.append(fila)
    cierres_revision.append({"ID_revision": identificador, "Decision": "CONSERVAR", "Filas": len(grupo)})

manual = pd.DataFrame(manual_aprobado, columns=columnas + ["_row_id"])
validar_salida(manual)
autores_unam_parcial = pd.concat([autores_unam_automatico, manual], ignore_index=True)
validar_salida(autores_unam_parcial)

# No presentar una base incompleta como final. No deduplicar ni renumerar.
autores_unam_separados = None
if not ids_pendientes:
    autores_unam_separados = autores_unam_parcial[columnas].copy().reset_index(drop=True)
    preparar_csv(autores_unam_separados)
    print("Revisión cerrada. Base completa disponible EN MEMORIA:", autores_unam_separados.shape)
else:
    print("Revisión pendiente de", len(ids_pendientes), "registros. Base final no disponible.")

Revisión pendiente de 663 registros. Base final no disponible.


## 16. RESUMEN FINAL

In [17]:
# Únicamente estos TRES CSV. No se crean más archivos ni subcarpetas.
os.makedirs(carpeta_salida, exist_ok=True)
guardar_csv(clasificacion_final, archivo_clasificacion)
guardar_csv(casos_revision, revision_manual)
# Se guarda aunque haya casos pendientes. No incluye filas aprobadas manualmente.
automatico_guardado = guardar_csv(salida_automatica_df, salida_automatica)

hash_despues = sha256(archivo)
if hash_despues != hash_antes:
    raise ValueError("El archivo original cambió.")
print("\n=== RESUMEN FINAL ===")
print("Filas de entrada:", len(df))
print("Registros originales para revisión:", len(filas_revision))
print("  Casos grandes (>20 autores O afiliaciones):", int(df["_caso_grande"].sum()))
print("  Separación no resuelta:", len(filas_pendientes))
print("  Más de dos afiliaciones UNAM:", len(filas_mas_dos))
print("Recuperadas desde otra representación:", recuperadas_otras)
print("Recuperadas con evidencia anterior:", recuperadas_previas)
print("Filas UNAM automáticas:", len(salida_automatica_df))
print("Filas UNAM manuales aprobadas (en memoria):", len(manual))
print("Relaciones externas excluidas automáticamente:", int(relaciones_automaticas["Estado_UNAM"].eq("EXTERNO").sum()))
print("Afiliaciones externas retiradas de autores conservados:", sum(
    r["Clasificaciones"].count("EXTERNO") for _, r in relaciones_unam.iterrows()
))
print("Revisiones pendientes:", len(ids_pendientes))
print("Columnas originales de la salida automática:", len(salida_automatica_df.columns))
print("Archivo original intacto:", hash_antes == hash_despues)
print("Clasificación:", archivo_clasificacion)
print("Casos manuales:", revision_manual)
if automatico_guardado:
    print("Salida automática guardada/verificada:", salida_automatica)
else:
    print("Salida automática NO actualizada: activa actualizar_archivos para regenerarla.")
print("No se generaron archivos adicionales.")

Guardado y releído: ../04_Limpieza/01_Internos_unam/clasificacion_afiliaciones_unam.csv (2580, 3)
Guardado y releído: ../04_Limpieza/01_Internos_unam/casos_revision_manual.csv (663, 27)
Guardado y releído: ../04_Limpieza/01_Internos_unam/autores_unam_separados_automatico.csv (3849, 15)

=== RESUMEN FINAL ===
Filas de entrada: 2582
Registros originales para revisión: 663
  Casos grandes (>20 autores O afiliaciones): 34
  Separación no resuelta: 189
  Más de dos afiliaciones UNAM: 0
Recuperadas desde otra representación: 618
Recuperadas con evidencia anterior: 0
Filas UNAM automáticas: 3849
Filas UNAM manuales aprobadas (en memoria): 0
Relaciones externas excluidas automáticamente: 4593
Afiliaciones externas retiradas de autores conservados: 148
Revisiones pendientes: 663
Columnas originales de la salida automática: 15
Archivo original intacto: True
Clasificación: ../04_Limpieza/01_Internos_unam/clasificacion_afiliaciones_unam.csv
Casos manuales: ../04_Limpieza/01_Internos_unam/casos_rev